<!-- OPS:MAT-02:inicio -->
## Esto es “completo” hoy (55 min)

Marca con lo que realmente ocurrió. No hace falta terminar todo.

- [ ] El entorno funciona o abrí `la ruta de rescate docente (no incluida)` y lo declaré.
- [ ] Producto mínimo: GET /health responde y existe un caso de abstención fuera del soporte. Sin model card.
- [ ] Si algo falló: una frase del error, no el traceback completo.
- [ ] Bitácora escrita, aunque diga `lab incompleto`.

Completo ≠ entendido del todo. Incompleto ≠ no puedes seguir. Si la instalación falla después de 10 minutos, usa `la ruta de rescate docente (no incluida)`, conserva el artefacto y marca incompleto.
**Extensión:** Si queda tiempo: metadata completa, tests adicionales, Docker y recorrido del prototipo.
<!-- OPS:MAT-02:fin -->

# L12 · Servir un estimador causal

**S12 · Lectura: Cap. 15 · 40–55 min**

## La pregunta

¿Qué debe devolver un servicio cuando la entrada está fuera de soporte?

## Qué harás

Separar la lógica de inferencia del transporte HTTP.

- Lee la idea breve.
- Ejecuta la demostración y mira el gráfico.
- Completa las celdas marcadas.
- Escribe qué significa el resultado y qué no permite afirmar.

**Producto:** Contrato local con health, metadata, predict y abstención.

> **Idea clave:** El servicio no mejora la identificación: solo debe conservarla y hacer sus límites visibles.

La solución está en `la ruta de rescate docente (no incluida)`. Intenta primero el notebook del estudiante.

In [ ]:
# Localiza lab_common.py (carpeta actual o una superior)
import sys, pathlib
for _p in [pathlib.Path.cwd(), *list(pathlib.Path.cwd().parents)[:3]]:
    if (_p / "lab_common.py").exists():
        sys.path.insert(0, str(_p)); break

## 0. Preparación

In [ ]:
from lab_common import *
import numpy as np, pandas as pd
from pydantic import BaseModel, Field, ValidationError, field_validator
from sklearn.ensemble import ExtraTreesRegressor
rng = setup()
art = a2_fit()
print("artefacto A2:", art["estimando"], "·", fmt_ci(art["theta"], art["se"]))
print("covariables del entrenamiento:", len(art["covariables"]))

## 1. Un servicio = lógica pura + transporte (§15.1)
**Regla del capítulo:** la decisión estadística vive en una función **pura** (`predict_effect`) que no sabe nada de HTTP; el servidor solo traduce entrada/salida. Así la función se prueba sin red, y el servicio y el entrenamiento comparten exactamente la misma lógica (menos *skew*).

Para que el orden de columnas importe, el servicio sirve un **CATE lineal** (un R-learner con ridge sobre residuos con cross-fitting), no solo el ATE.

In [ ]:
sc = load_scenario("synthetic/inference/ihdp_medium")
cols = list(art["covariables"])
X, Y, T = sc.xyt("train")
mk = lambda: ExtraTreesRegressor(n_estimators=40, min_samples_leaf=5, max_features=0.8, random_state=0)
n = len(Y); ry = np.zeros(n); rd = np.zeros(n)
for te in kfold(n, 5, SEED):
    tr = np.setdiff1d(np.arange(n), te)
    ry[te] = Y[te] - mk().fit(X[tr], Y[tr]).predict(X[te]); rd[te] = T[te] - mk().fit(X[tr], T[tr]).predict(X[te])
mu_x, sd_x = X.mean(0), X.std(0) + 1e-9
Z = (X - mu_x) / sd_x
A = np.column_stack([rd, rd[:, None] * Z])                       # ry ≈ rd * (a + z'b)
lam = np.r_[0, np.full(Z.shape[1], 5.0)]
coef = np.linalg.solve(A.T @ A + np.diag(lam), A.T @ ry)
modelo = {"a": float(coef[0]), "b": coef[1:].tolist(), "mu": mu_x.tolist(), "sd": sd_x.tolist()}
print("τ(x) = a + z'b ·  a =", round(modelo["a"], 3), "· mayor |b|:", cols[int(np.argmax(np.abs(coef[1:])))], round(float(np.abs(coef[1:]).max()), 3))

In [ ]:
# Vista rápida de los datos del escenario
import matplotlib.pyplot as plt
import numpy as np
_fig, _ax = plt.subplots(1, 2, figsize=(9, 3.2))
try:
    _scenario = globals().get("sc") or globals().get("lc")
    if _scenario is None:
        raise NameError("No hay un escenario cargado para el gráfico editorial")
    _df = _scenario.train
    if _scenario.treatment is not None and _scenario.outcome is not None:
        _t = _df[_scenario.treatment].to_numpy(float)
        _y = _df[_scenario.outcome].to_numpy(float)
        if np.unique(_t).size == 2:
            _counts = [int((_t == value).sum()) for value in (0, 1)]
            _ax[0].bar(["control", "tratado"], _counts)
            _ax[0].set_title("Asignación")
            _ax[1].boxplot([_y[_t == 0], _y[_t == 1]], labels=["control", "tratado"])
            _ax[1].set_title("Resultado observado")
        else:
            _ax[0].hist(_t, bins=20)
            _ax[0].set_title(_scenario.treatment)
            _ax[1].scatter(_t, _y, s=8, alpha=.35)
            _ax[1].set_title(f"{_scenario.treatment} frente a {_scenario.outcome}")
    elif _scenario.adj is not None:
        _ax[0].imshow(_scenario.adj, cmap="Greys", interpolation="nearest")
        _ax[0].set_title("Grafo del escenario")
        _ax[1].axis("off")
    else:
        _ax[0].axis("off"); _ax[1].axis("off")
    _fig.suptitle(f"Escenario: {_scenario.name}")
    _fig.tight_layout(); plt.show()
finally:
    plt.close(_fig)

In [ ]:
def predict_effect(modelo, columnas, x):
    """LÓGICA PURA. `x` es un vector en el ORDEN de `columnas` (el del entrenamiento)."""
    z = (np.asarray(x, float) - np.asarray(modelo["mu"])) / np.asarray(modelo["sd"])
    return float(modelo["a"] + z @ np.asarray(modelo["b"]))
print("τ̂ de la primera fila de validación:", round(predict_effect(modelo, cols, sc.val[cols].iloc[0].to_numpy()), 3))

## 2. Skew: el error que no da error (§15.2)
Si el servicio recibe una **lista** y la ordena distinto que el entrenamiento, el modelo responde algo plausible pero **equivocado**, sin ningún aviso. Lo provocamos intercambiando dos columnas.

In [ ]:
V = sc.val[cols].to_numpy(float)
bien = np.array([predict_effect(modelo, cols, r) for r in V])
swap = list(range(len(cols))); swap[0], swap[1] = swap[1], swap[0]                    # el cliente envía x1 donde va x0 y viceversa
mal = np.array([predict_effect(modelo, cols, r[swap]) for r in V])
print(f"diferencia media absoluta entre la respuesta correcta y la del cliente con columnas intercambiadas: {np.abs(bien - mal).mean():.3f}")
print(f"(el CATE medio es {bien.mean():.3f}; no hubo ninguna excepción)")

**Mira el resultado.** ¿Por qué las pruebas unitarias de `predict_effect` no atrapan este problema, y qué lo evita?

**Respuesta:**

`predict_effect` hace exactamente lo que se le pide: el defecto está en **el contrato** (cómo se identifica cada valor). Se evita aceptando **diccionarios con nombres** y reordenando con la lista de covariables del artefacto, y rechazando columnas faltantes o sobrantes. Es la misma razón por la que el `/metadata` expone la lista y el orden de las características.



## 3. El contrato (§15.3)
La entrada es un diccionario `{nombre: valor}` y la salida incluye `abstain` y `reason`. Usamos **Pydantic** para validar.

### Completa tú
1. `PredictRequest` con `features: dict[str, float]`, que **exija** exactamente las covariables del artefacto (ni falta ni sobra ninguna) y valores finitos.
2. `PredictResponse` con `effect` (opcional si se abstiene), `ate_ic` (lista de 2), `abstain: bool`, `reason` (opcional), `model_version`.
3. `support_check(features, soporte, margen=0.05)` → lista de columnas **fuera de soporte** (valor fuera de [min, max] del entrenamiento ampliado un `margen` relativo del rango).

In [ ]:
# ✏️ Completa tú — contrato y support_check
# Pista: Pydantic v2: usa @field_validator('features') y lanza ValueError con un mensaje claro.
COLS = list(cols)

class PredictRequest(BaseModel):
    features: dict[str, float]
    # TODO: validador que exija exactamente COLS y valores finitos

class PredictResponse(BaseModel):
    pass
    # TODO: effect (opcional), ate_ic (2 valores), abstain, reason (opcional), model_version

def support_check(features, soporte, margen=0.05):
    """Lista de columnas cuyo valor cae fuera de [min - margen·rango, max + margen·rango]."""
    raise NotImplementedError("completa support_check")

In [ ]:
fila_ok = sc.val[cols].iloc[0].to_dict()
PredictRequest(features=fila_ok)                               # válida
casos = {"falta una columna": {k: v for k, v in list(fila_ok.items())[1:]},
         "columna sobrante": {**fila_ok, "extra": 1.0},
         "valor no finito": {**fila_ok, cols[0]: float("nan")}}
for nombre, f in casos.items():
    try:
        PredictRequest(features=f); print("✘ debió rechazar:", nombre)
    except ValidationError as e:
        print("✔ rechaza:", nombre)
lab_check("support_check", lambda: (support_check(fila_ok, art["soporte"]) == [],
                                    support_check({**fila_ok, cols[0]: art["soporte"][cols[0]][1] + 100}, art["soporte"]) == [cols[0]]),
          [((), (True, True), 0)])

## 4. El servicio: tres rutas sin servidor (§15.4)
`Servicio` expone `health()`, `metadata()` y `predict(payload)` y devuelve `(código, cuerpo)`. Es lo que FastAPI envolvería; aquí lo probamos directamente, **sin red**. Construimos dos versiones: la que *no se abstiene* (como hace hoy el B1 del repositorio: devuelve siempre el ATE) y la **envuelta** (abstiene fuera de soporte).

In [ ]:
class Servicio:
    def __init__(self, abstiene: bool, version="1.0"):
        self.abstiene, self.version = abstiene, version
    def health(self): return 200, {"status": "ok"}
    def metadata(self):
        return 200, {"model_version": self.version, "features": COLS, "n_features": len(COLS), "estimando": art["estimando"],
                     "hash_datos_entrenamiento": art["hash_datos_entrenamiento"]}
    def predict(self, payload: dict):
        try:
            req = PredictRequest(**payload)
        except ValidationError as e:
            return 422, {"detail": str(e).splitlines()[0]}
        x = [req.features[c] for c in COLS]                               # reordena por NOMBRE
        fuera = support_check(req.features, art["soporte"]) if self.abstiene else []
        if fuera:
            r = PredictResponse(effect=None, ate_ic=art["ic95"], abstain=True, reason=f"fuera de soporte: {fuera[:3]}", model_version=self.version)
        else:
            r = PredictResponse(effect=predict_effect(modelo, COLS, x), ate_ic=art["ic95"], abstain=False, model_version=self.version)
        return 200, r.model_dump()
srv = Servicio(abstiene=True)
print(srv.health()); print(srv.metadata()[1]["n_features"], "covariables en /metadata")
print(srv.predict({"features": fila_ok}))

## 5. Cinco pruebas de humo (§15.5)
Mínimas, sobre la app *real* y con **entradas válidas de una fila real** (no ceros). Se ejecutan contra las dos versiones: la que no se abstiene **falla la prueba 5**, y esa es la brecha que el curso documenta.

In [ ]:
def humo(s: Servicio):
    res = {}
    c, b = s.health(); res["1 /health responde 200"] = (c == 200 and b["status"] == "ok")
    c, b = s.metadata(); res["2 /metadata trae las covariables del entrenamiento"] = (b["features"] == list(art["covariables"]) and b["n_features"] == len(art["covariables"]))
    c, b = s.predict({"features": fila_ok}); res["3 /predict con una fila real devuelve un efecto finito"] = (c == 200 and b["effect"] is not None and np.isfinite(b["effect"]))
    c, b = s.predict({"features": {k: v for k, v in list(fila_ok.items())[1:]}}); res["4 /predict rechaza una entrada incompleta (422)"] = (c == 422)
    raro = {**fila_ok, cols[0]: art["soporte"][cols[0]][1] + 100 * (art["soporte"][cols[0]][1] - art["soporte"][cols[0]][0] + 1)}
    c, b = s.predict({"features": raro}); res["5 /predict se abstiene fuera de soporte"] = (c == 200 and b["abstain"] is True)
    return res
tabla = pd.DataFrame({"sin abstención (como B1)": humo(Servicio(abstiene=False)), "con envoltorio": humo(Servicio(abstiene=True))})
show(tabla.replace({True: "pasa", False: "FALLA"}))

**Mira el resultado.** La prueba 5 falla en la versión sin abstención. ¿Por qué es peligroso que un servicio devuelva un efecto fuera de soporte «con total confianza»?

**Respuesta:**

Porque el modelo **extrapola**: ante una fila que no se parece a ninguna del entrenamiento, el resultado ya no tiene respaldo (el supuesto de solapamiento y la validez del modelo no se sostienen) pero llega con la misma apariencia que uno fiable. Abstenerse con una razón legible hace visible el límite; es la decisión *abstenerse* del vocabulario del curso.



## 6. FastAPI (opcional, sin probar en el sandbox del autor)
Esta celda envuelve el mismo `Servicio` con FastAPI y lo prueba con `httpx.ASGITransport` (sin abrir puertos). Solo corre si tienes `fastapi` y `httpx`; si no, lo avisa.

In [ ]:
try:
    import asyncio, httpx
    from fastapi import FastAPI, HTTPException
    app = FastAPI(title="Servicio causal (lab)")
    S = Servicio(abstiene=True)
    @app.get("/health")
    def _h(): return S.health()[1]
    @app.get("/metadata")
    def _m(): return S.metadata()[1]
    @app.post("/predict", response_model=PredictResponse)
    def _p(req: PredictRequest):
        c, b = S.predict({"features": req.features})
        return b
    async def probar():
        async with httpx.AsyncClient(transport=httpx.ASGITransport(app=app), base_url="http://t") as cl:
            r = await cl.post("/predict", json={"features": fila_ok}); return r.status_code, r.json()
    try:
        asyncio.get_running_loop()
    except RuntimeError:
        print(asyncio.run(probar()))
    else:
        print(await probar())
except ImportError as e:
    print("FastAPI/httpx no están instalados (", e.name, "): celda omitida. La lógica y el contrato ya se probaron arriba.")

## 7. Soporte y supuesto

In [ ]:
# EVALUATION-CELL
tau_v = sc.tau("val")
if tau_v is not None:
    print("correlación entre el CATE servido y el verdadero (validación):", round(float(np.corrcoef(bien, tau_v)[0, 1]), 3))
print("covariables del servicio == covariables del entrenamiento:", srv.metadata()[1]["features"] == list(art["covariables"]))

In [ ]:
iguales = srv.metadata()[1]["features"] == list(art["covariables"])
fila_b = bitacora(
    pregunta="¿El servicio responde con las mismas covariables (nombre y orden) que el entrenamiento y se abstiene fuera de soporte?",
    metodo="Servicio con contrato Pydantic, reordenamiento por nombre y chequeo de soporte; cinco pruebas de humo",
    estimando="CATE de una fila (R-learner lineal) con el ATE de A2 como contexto",
    numero_ic=fmt_ci(art["theta"], art["se"]),
    supuesto=supuesto("observable", "medido", f"las {len(cols)} covariables del servicio coinciden con las del entrenamiento: {iguales}"),
    decision="continuar" if iguales else "alertar", lab="L12")

---
## Qué queda para el libro
Quedan en el libro: el `Dockerfile`, el checklist completo de despliegue y la configuración de FastAPI/uvicorn. En este sandbox FastAPI y httpx no estaban instalados: la celda opcional de FastAPI está escrita pero **no se probó aquí**.

*Material formativo (sin nota). Los datos son sintéticos: ver la ficha del escenario.*